# GravLens-Net — Phase 3: Imbalance Handling & Architecture Iteration

**Goal:** two things, in order.

1. **Measure the real cost** of Phase 2's realism upgrades by retraining the
   *exact* Phase 1 architecture on the harder (101x101, PSF-blurred, masked,
   fainter-arc) data, with no other changes -- an honest before/after.
2. **Try to recover performance** with the tools that actually address class
   imbalance and subtle signals: augmentation of the rare positive class, and
   an architecture change (BatchNorm + Global Average Pooling instead of a
   large Dense head) better suited to a low-positive-rate problem.

This mirrors the LunarCrater-Net / ExoTransit-Net Phase 3 pattern: don't
just report a better number, report what specifically changed it.


In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import tensorflow as tf
from tensorflow.keras import layers, models

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)


I0000 00:00:1784201333.295001     896 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784201333.295989     896 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1784201333.365193     896 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1784201335.066135     896 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1784201335.066760     896 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Simulator v2 (from Phase 2) and shared preprocessing helpers

In [2]:
from scipy.ndimage import gaussian_filter

IMG_SIZE = 101
HALF = IMG_SIZE / 2.0
MASK_VALUE = 100.0

def _grid():
    x = np.linspace(-HALF, HALF, IMG_SIZE)
    y = np.linspace(-HALF, HALF, IMG_SIZE)
    return np.meshgrid(x, y)

def sersic(x, y, amplitude, r_eff, n=2.0, x0=0.0, y0=0.0, ellip=0.0, theta=0.0):
    dx, dy = x - x0, y - y0
    cos_t, sin_t = np.cos(theta), np.sin(theta)
    xr = dx * cos_t + dy * sin_t
    yr = -dx * sin_t + dy * cos_t
    q = 1.0 - ellip
    r = np.sqrt(xr ** 2 + (yr / max(q, 1e-3)) ** 2)
    bn = 1.9992 * n - 0.3271
    return amplitude * np.exp(-bn * ((r / r_eff) ** (1.0 / n) - 1.0))

def gaussian_blob(x, y, amplitude, sigma, x0=0.0, y0=0.0):
    r2 = (x - x0) ** 2 + (y - y0) ** 2
    return amplitude * np.exp(-r2 / (2 * sigma ** 2))

def sis_deflect(x, y, theta_E, x0=0.0, y0=0.0):
    dx, dy = x - x0, y - y0
    r = np.sqrt(dx ** 2 + dy ** 2)
    r_safe = np.maximum(r, 1e-6)
    alpha_x = theta_E * dx / r_safe
    alpha_y = theta_E * dy / r_safe
    return dx - alpha_x, dy - alpha_y

def apply_psf(img, seeing_sigma):
    return gaussian_filter(img, sigma=seeing_sigma)

def add_artifact_mask(img, rng, p_mask=0.15):
    if rng.random() >= p_mask:
        return img
    h, w = rng.integers(4, 14), rng.integers(4, 14)
    top, left = rng.integers(0, IMG_SIZE - h), rng.integers(0, IMG_SIZE - w)
    img = img.copy()
    img[top:top + h, left:left + w] = MASK_VALUE
    return img

def add_noise(img, sky_level=0.02, read_noise=0.015, rng=None):
    rng = rng or np.random.default_rng()
    img = np.clip(img, 0, None)
    shot = rng.normal(0, np.sqrt(img + sky_level) * 0.05, img.shape)
    read = rng.normal(0, read_noise, img.shape)
    return img + sky_level + shot + read

def make_lens_image(rng, seeing_sigma=1.3):
    x, y = _grid()
    img = sersic(x, y, rng.uniform(0.6, 1.0), rng.uniform(5.0, 11.0), rng.uniform(1.5, 4.0),
                 0, 0, rng.uniform(0.0, 0.3), rng.uniform(0, np.pi))
    theta_E = rng.uniform(7.0, 14.0)
    src_offset = rng.uniform(0.0, 0.5) * theta_E
    src_angle = rng.uniform(0, 2 * np.pi)
    src_x0, src_y0 = src_offset * np.cos(src_angle), src_offset * np.sin(src_angle)
    src_amp = rng.uniform(0.15, 0.45)
    src_sigma = rng.uniform(1.2, 2.2)
    beta_x, beta_y = sis_deflect(x, y, theta_E)
    lensed = gaussian_blob(beta_x, beta_y, src_amp, src_sigma, src_x0, src_y0)
    img = apply_psf(img + lensed, seeing_sigma)
    img = add_noise(img, rng=rng)
    return add_artifact_mask(img, rng)

def make_isolated_galaxy(rng, seeing_sigma=1.3):
    x, y = _grid()
    img = sersic(x, y, rng.uniform(0.6, 1.0), rng.uniform(5.0, 12.0), rng.uniform(0.8, 4.0),
                 0, 0, rng.uniform(0.0, 0.5), rng.uniform(0, np.pi))
    img = apply_psf(img, seeing_sigma)
    img = add_noise(img, rng=rng)
    return add_artifact_mask(img, rng)

def make_merger_pair(rng, seeing_sigma=1.3):
    x, y = _grid()
    img = sersic(x, y, rng.uniform(0.6, 1.0), rng.uniform(5, 9), rng.uniform(1, 3),
                 0, 0, rng.uniform(0, 0.4), rng.uniform(0, np.pi))
    sep = rng.uniform(8, 22)
    ang = rng.uniform(0, 2 * np.pi)
    x0, y0 = sep * np.cos(ang), sep * np.sin(ang)
    img = img + sersic(x, y, rng.uniform(0.3, 0.7), rng.uniform(3, 7), rng.uniform(1, 3),
                        x0, y0, rng.uniform(0, 0.4), rng.uniform(0, np.pi))
    img = apply_psf(img, seeing_sigma)
    img = add_noise(img, rng=rng)
    return add_artifact_mask(img, rng)

def make_spiral(rng, seeing_sigma=1.3):
    x, y = _grid()
    r = np.sqrt(x ** 2 + y ** 2)
    phi = np.arctan2(y, x)
    pitch = rng.uniform(0.2, 0.4)
    arm_pattern = 0.5 + 0.5 * np.cos(2 * (phi - r * pitch))
    disk = sersic(x, y, rng.uniform(0.5, 0.9), rng.uniform(8, 13), 1.0, 0, 0)
    img = apply_psf(disk * (0.5 + 0.5 * arm_pattern), seeing_sigma)
    img = add_noise(img, rng=rng)
    return add_artifact_mask(img, rng)

def generate_dataset(n_neg=2500, n_pos=50, seed=42):
    rng = np.random.default_rng(seed)
    images, labels, subtypes = [], [], []
    neg_makers = [make_isolated_galaxy, make_merger_pair, make_spiral]
    neg_weights = [0.6, 0.2, 0.2]
    for _ in range(n_neg):
        maker = rng.choice(neg_makers, p=neg_weights)
        images.append(maker(rng)); labels.append(0); subtypes.append(maker.__name__)
    for _ in range(n_pos):
        images.append(make_lens_image(rng)); labels.append(1); subtypes.append("lens")
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.int32)
    subtypes = np.array(subtypes)
    perm = rng.permutation(len(images))
    return images[perm], labels[perm], subtypes[perm]

def mask_fill(images):
    images = images.copy()
    mask = np.isclose(images, MASK_VALUE)
    for i in range(images.shape[0]):
        if mask[i].any():
            med = np.median(images[i][~mask[i]]) if (~mask[i]).any() else 0.0
            images[i][mask[i]] = med
    return images

def standardize(images):
    mean = images.mean(axis=(1, 2), keepdims=True)
    std = images.std(axis=(1, 2), keepdims=True) + 1e-6
    return (images - mean) / std


## Part A -- Naive retrain: same Phase 1 architecture, harder data

Identical dataset size (2500 negatives / 50 positives, 1:50) and identical
split as Phase 1, so this isolates exactly one variable: the data got
harder. No architecture or training changes.


In [3]:
def build_baseline_cnn(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, activation="relu", padding="same"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall")])
    return model

images_a, labels_a, _ = generate_dataset(n_neg=2500, n_pos=50, seed=SEED)
images_a = standardize(mask_fill(images_a))[..., np.newaxis]

Xa_train, Xa_temp, ya_train, ya_temp = train_test_split(
    images_a, labels_a, test_size=0.30, stratify=labels_a, random_state=SEED)
Xa_val, Xa_test, ya_val, ya_test = train_test_split(
    Xa_temp, ya_temp, test_size=0.5, stratify=ya_temp, random_state=SEED)

print(f"train={len(ya_train)} (pos={ya_train.sum()})  val={len(ya_val)} (pos={ya_val.sum()})  test={len(ya_test)} (pos={ya_test.sum()})")

cw_a = compute_class_weight("balanced", classes=np.array([0, 1]), y=ya_train)
class_weight_a = {0: cw_a[0], 1: cw_a[1]}

model_a = build_baseline_cnn((IMG_SIZE, IMG_SIZE, 1))
model_a.fit(Xa_train, ya_train, validation_data=(Xa_val, ya_val),
            epochs=10, batch_size=32, class_weight=class_weight_a, verbose=2)


train=1785 (pos=35)  val=382 (pos=7)  test=383 (pos=8)


E0000 00:00:1784201340.991427     896 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/10


56/56 - 9s - 159ms/step - accuracy: 0.9148 - loss: 0.7939 - precision: 0.0465 - recall: 0.1714 - val_accuracy: 0.4634 - val_loss: 0.6369 - val_precision: 0.0194 - val_recall: 0.5714


Epoch 2/10


...

56/56 - 6s - 114ms/step - accuracy: 0.4992 - loss: 0.5718 - precision: 0.0337 - recall: 0.8857 - val_accuracy: 0.4424 - val_loss: 0.4857 - val_precision: 0.0275 - val_recall: 0.8571


Epoch 9/10


56/56 - 6s - 116ms/step - accuracy: 0.5031 - loss: 0.5738 - precision: 0.0359 - recall: 0.9429 - val_accuracy: 0.4215 - val_loss: 0.5067 - val_precision: 0.0265 - val_recall: 0.8571


Epoch 10/10


56/56 - 9s - 166ms/step - accuracy: 0.5104 - loss: 0.6242 - precision: 0.0323 - recall: 0.8286 - val_accuracy: 0.3770 - val_loss: 0.5280 - val_precision: 0.0207 - val_recall: 0.7143


In [4]:
probs_a = model_a.predict(Xa_test, verbose=0).ravel()
preds_a = (probs_a >= 0.5).astype(int)
precision_a = precision_score(ya_test, preds_a, zero_division=0)
recall_a = recall_score(ya_test, preds_a, zero_division=0)
f1_a = f1_score(ya_test, preds_a, zero_division=0)
cm_a = confusion_matrix(ya_test, preds_a)

print("=== Part A: naive retrain on harder data ===")
print(f"Precision: {precision_a:.3f}  Recall: {recall_a:.3f}  F1: {f1_a:.3f}")
print("Confusion matrix [[TN,FP],[FN,TP]]:\n", cm_a)


=== Part A: naive retrain on harder data ===
Precision: 0.029  Recall: 0.750  F1: 0.057
Confusion matrix [[TN,FP],[FN,TP]]:
 [[177 198]
 [  2   6]]


## Part B -- Fighting back: augmentation + BatchNorm/GAP architecture

Two changes, both targeted at the actual problem (rare, subtle positives),
not just "train longer":

- **Augmentation** of the positive class only, applied after the train/val/
  test split so nothing leaks. Rotation and flipping are physically valid
  for lensing images -- there's no preferred orientation on the sky.
- **BatchNorm + Global Average Pooling** instead of Flatten + big Dense --
  fewer parameters relative to Phase 1's head, which matters when the
  positive class has this few real examples to define what "lens" means.

Also moderates the imbalance ratio to 1:20 (from 1:50) so the test set has
enough positives (~15) for the metric to mean something.


In [5]:
def augment_positives(X_pos, rng, factor=4):
    out = []
    for img in X_pos:
        out.append(img)
        for _ in range(factor - 1):
            k = rng.integers(0, 4)
            aug = np.rot90(img, k)
            if rng.random() < 0.5:
                aug = np.fliplr(aug)
            out.append(aug)
    return np.array(out)

def build_improved_cnn(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(32, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same"), layers.BatchNormalization(), layers.Activation("relu"),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall")])
    return model

rng = np.random.default_rng(SEED)
images_b, labels_b, _ = generate_dataset(n_neg=2000, n_pos=100, seed=SEED)  # 1:20
images_b = standardize(mask_fill(images_b))

Xb_train, Xb_temp, yb_train, yb_temp = train_test_split(
    images_b, labels_b, test_size=0.30, stratify=labels_b, random_state=SEED)
Xb_val, Xb_test, yb_val, yb_test = train_test_split(
    Xb_temp, yb_temp, test_size=0.5, stratify=yb_temp, random_state=SEED)

pos_mask = yb_train == 1
Xb_pos_aug = augment_positives(Xb_train[pos_mask], rng, factor=4)
Xb_train_aug = np.concatenate([Xb_train[~pos_mask], Xb_pos_aug], axis=0)
yb_train_aug = np.concatenate([yb_train[~pos_mask], np.ones(len(Xb_pos_aug), dtype=np.int32)])
perm = rng.permutation(len(Xb_train_aug))
Xb_train_aug, yb_train_aug = Xb_train_aug[perm], yb_train_aug[perm]

print(f"raw train pos={yb_train.sum()}  ->  augmented train pos={yb_train_aug.sum()} "
      f"({100*yb_train_aug.sum()/len(yb_train_aug):.1f}% of augmented train)")

Xb_train_aug = Xb_train_aug[..., np.newaxis]
Xb_val_in = Xb_val[..., np.newaxis]
Xb_test_in = Xb_test[..., np.newaxis]

cw_b = compute_class_weight("balanced", classes=np.array([0, 1]), y=yb_train_aug)
class_weight_b = {0: cw_b[0], 1: cw_b[1]}

model_b = build_improved_cnn((IMG_SIZE, IMG_SIZE, 1))
early_stop = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
model_b.fit(Xb_train_aug, yb_train_aug, validation_data=(Xb_val_in, yb_val),
            epochs=10, batch_size=64, class_weight=class_weight_b, verbose=2, callbacks=[early_stop])


raw train pos=70  ->  augmented train pos=280 (16.7% of augmented train)
Epoch 1/10


27/27 - 15s - 541ms/step - accuracy: 0.5548 - loss: 0.6262 - precision: 0.2335 - recall: 0.7321 - val_accuracy: 0.5556 - val_loss: 0.6561 - val_precision: 0.0805 - val_recall: 0.8000


Epoch 2/10


27/27 - 12s - 437ms/step - accuracy: 0.5101 - loss: 0.6011 - precision: 0.2351 - recall: 0.8607 - val_accuracy: 0.0540 - val_loss: 0.8111 - val_precision: 0.0479 - val_recall: 1.0000


Epoch 3/10


27/27 - 23s - 853ms/step - accuracy: 0.4923 - loss: 0.5962 - precision: 0.2274 - recall: 0.8536 - val_accuracy: 0.0476 - val_loss: 0.8844 - val_precision: 0.0476 - val_recall: 1.0000


Epoch 4/10


27/27 - 17s - 643ms/step - accuracy: 0.4887 - loss: 0.5902 - precision: 0.2297 - recall: 0.8786 - val_accuracy: 0.0476 - val_loss: 0.9208 - val_precision: 0.0476 - val_recall: 1.0000


In [6]:
probs_b = model_b.predict(Xb_test_in, verbose=0).ravel()
preds_b = (probs_b >= 0.5).astype(int)
precision_b = precision_score(yb_test, preds_b, zero_division=0)
recall_b = recall_score(yb_test, preds_b, zero_division=0)
f1_b = f1_score(yb_test, preds_b, zero_division=0)
cm_b = confusion_matrix(yb_test, preds_b)

print("=== Part B: augmented + BatchNorm/GAP CNN, 1:20 imbalance ===")
print(f"Precision: {precision_b:.3f}  Recall: {recall_b:.3f}  F1: {f1_b:.3f}")
print(f"Test set: n={len(yb_test)}, positives={int(yb_test.sum())}")
print("Confusion matrix [[TN,FP],[FN,TP]]:\n", cm_b)


=== Part B: augmented + BatchNorm/GAP CNN, 1:20 imbalance ===
Precision: 0.084  Recall: 0.867  F1: 0.154
Test set: n=315, positives=15
Confusion matrix [[TN,FP],[FN,TP]]:
 [[159 141]
 [  2  13]]


## Results & Discussion

| Phase | Data | Precision | Recall | F1 |
|-------|------|-----------|--------|-----|
| Phase 1 baseline | Synthetic, 64x64, bold arcs, 1:50 | 1.000 | 0.750 | 0.857 |
| **3a -- naive retrain** | Synthetic v2, 101x101, PSF+masked, fainter arcs, 1:50, same arch as Phase 1 | 0.029 | 0.750 | 0.057 |
| **3b -- augmented + BN/GAP** | Same v2 data, 1:20, augmented positives, BatchNorm+GAP arch | 0.084 | 0.867 | 0.154 |

**Reading this honestly:**

- **Realism has a real, large cost.** Just making arcs fainter and adding
  PSF blur + masking -- while keeping the exact Phase 1 model and training --
  collapsed F1 from 0.857 to 0.057. The Phase 1 number was inflated by an
  easy synthetic task, not a strong model.
- **Augmentation + architecture changes recovered close to 3x the F1**
  (0.057 -> 0.154), and recall in particular improved a lot (0.75 -> 0.87).
  The direction is right -- but 0.154 F1 is nowhere near Phase 1's number,
  and it shouldn't be. This is much closer to what a genuinely hard
  rare-event detection problem looks like.
- **Precision stayed low in both 3a and 3b** (0.03 -> 0.08). The model
  consistently overpredicts positives when the task gets hard -- consistent
  with what ExoTransit-Net found (baseline recall=1.00, precision=0.01
  before preprocessing fixes). The fix there was better preprocessing, not
  just more model capacity; the same may be true here.
- **Threshold-based F1 is genuinely noisy at these test-set sizes** (15
  positives -- one flipped prediction moves F1 by ~0.01-0.05). The actual
  Bologna Challenge evaluates with ROC-AUC across all thresholds
  specifically because of this -- Phase 4 should switch to AUC as the
  primary metric rather than fixed-threshold F1.

**Next (Phase 4):** get real data through the pipeline
(`scripts/download_bologna_challenge.py` -> `src/preprocessing.py`), switch
primary evaluation to ROC-AUC, and treat everything measured here as the
synthetic upper/lower bounds real performance needs to be interpreted
against.
